In [1]:
#load data
import pandas as pd

data = pd.read_csv("../../processed/final_predicted.csv")
print("length:", len(data), "peptides")

data.head()

length: 6327 peptides


,epitope_id,best_hla,label,mt_seq,wt_seq,mt_length,mt_affinity,mt_affinity_percentile,mt_presentation_score,mt_presentation_percentile,wt_affinity,wt_affinity_percentile,wt_presentation_score,wt_presentation_percentile,mt_presented,wt_presented,delta_affinity,delta_presentation_score,delta_log_affinity,group
0,551,HLA-A*02:01,1,ACDPHSGHFV,ARDPHSGHFV,10,3125.736683,2.796875,0.053443,4.807092,13379.793366,7.050250,0.029345,8.269212,0,0,-10254.056683,0.024097,-1.454076,notpresented_immunogenic
1,785,HLA-B*44:03,0,ADVEFCLSL,ANVEFCL,9,848.828242,0.972500,0.108789,2.760054,32406.910919,40.329625,0.004118,62.744674,0,0,-31558.082677,0.104670,-3.642270,notpresented_nonimmunogenic
2,3147,HLA-A*02:01,1,AMLGTHTMEV,AMLSPHAMDV,10,37.507257,0.260250,0.768811,0.326821,104.448134,0.625750,0.483023,0.858478,1,1,-66.940877,0.285789,-1.024156,presented_immunogenic
3,14672,HLA-A*01:01,1,EVDPIGHLY,EVVPISHLY,9,33.501012,0.007125,0.983180,0.004511,164.471650,0.231000,0.941244,0.060136,1,1,-130.970638,0.041936,-1.591163,presented_immunogenic
4,23214,HLA-A*02:01,1,GVYDGEEHSV,GAYDGEEH,10,46.603713,0.330000,0.871138,0.167038,29876.517551,41.398000,0.003482,99.286603,1,0,-29829.913838,0.867656,-6.463148,presented_immunogenic


In [2]:
#convert into python lists
mt_seqs = data["mt_seq"].tolist()
wt_seqs = data["wt_seq"].tolist()
labels = data["label"].values 

#also add epitope id
epitope_ids = data["epitope_id"].astype(str).tolist()

In [3]:
#set device
import torch 
import esm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#load esm2 model
model_name = "esm2_t33_650M_UR50D"
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D() 
#move model to cpu
model = model.to(device)
#turn off traning mode
model.eval()
#convert seq to tokens
batch_converter = alphabet.get_batch_converter()

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /Users/tamannatandon/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /Users/tamannatandon/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


In [ ]:
#embedding loop 
import numpy as np 

def embed_seq(sequences, batch_size=16): 
    per_residue = []
    bos_embeds = [] 
    eos_embeds = [] 

    for i in range(0, len(sequences), batch_size): 
        batch = sequences[i:i+batch_size]
        #label identifier
        batch_data = [(f"seq_{i+j}", seq) for j, seq in enumerate(batch)]

        #convert seqs to tokens
        _, _, tokens = batch_converter(batch_data)
        tokens = tokens.to(device)

        #run model and extract last layer
        with torch.no_grad(): 
            results = model(tokens, repr_layers=[33])
        #token reps shape
        token_reps = results["representations"][33]

        #process each sequence in batch 
        for j, seq in enumerate(batch): 
            seq_length = len(seq)

            #bos token
            bos = token_reps[j, 0].cpu().numpy().astype(np.float32)

            #eos token
            eos = token_reps[j, seq_length + 1].cpu().numpy().astype(np.float32)

            #per residue
            #esm2 adds bos at front and eds at end
            #so remove tokens to get residue 
            per_res = token_reps[j, 1:seq_length + 1].cpu().numpy().astype(np.float32)

            per_residue.append(per_res)
            bos_embeds.append(bos)
            eos_embeds.append(eos)
            

    return per_residue, np.array(bos_embeds), np.array(eos_embeds)
        

In [ ]:
#generate embeddings
mt_per_res, mt_bos, mt_eos = embed_seq(mt_seqs)

In [ ]:
#also for wt
wt_per_res, wt_bos, wt_eos = embed_seq(wt_seqs)

In [ ]:
#sanity checks

#check if all shapes are same 
print("mt_bos shape:", mt_bos.shape)
print("mt_eos shape:", mt_eos.shape)
print("wt_bos shape:", wt_bos.shape)
print("wt_eos shape:", wt_eos.shape)
print("mt_per_res shape:", len(mt_per_res))
print("wt_per_res shape:", len(wt_per_res))

#check for nans
print("NaNs in mt_bos:", np.isnan(mt_bos).sum())
print("NaNs in wt_bos:", np.isnan(wt_bos).sum())

mt_mean shape: (6327, 1280)
wt_mean shape: (6327, 1280)
mt_per_res shape: 6327
wt_per_res shape: 6327
NaNs in mt_mean: 0
NaNs in wt_mean: 0


In [ ]:
#save output 
save_dict = {
    #bos/eos vectors 
    "mt_bos": mt_bos,
    "mt_eos": mt_eos,
    "wt_bos": wt_bos,
    "wt_eos": wt_eos,

    #metadata
    "labels": labels.astype(np.int32), 
    "lengths": np.array([len(s) for s in mt_seqs], dtype=np.int32), 
    "n_peptides": np.array(len(mt_seqs), dtype=np.int32), 
    "epitope_ids": np.array(epitope_ids, dtype=object)
}

#add per residue assays 
#variable elngth, so store as separate arrays
for i in range(len(mt_seqs)): 
    save_dict[f"mt_perres_{i}"] = mt_per_res[i]
    save_dict[f"wt_perres_{i}"] = wt_per_res[i]

np.savez_compressed("esm2_embeddings.npz", **save_dict)